In [3]:
import os
import sys

# 1. Force the Environment Paths to point to Java 17
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

# 2. Initialize Spark with explicit configuration overrides
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Notebook-ETL-Prototyping") \
    .master("local[4]") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.executor.extraJavaOptions", "-Djava.security.manager=allow") \
    .config("spark.driver.extraJavaOptions", "-Djava.security.manager=allow") \
    .getOrCreate()

print("Spark Session Active!")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/02 17:24:43 WARN Utils: Your hostname, LAPTOP-PS6VFB9A, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/02 17:24:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/abhishek/dsba2026/dsba/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/02 17:24:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Session Active!


In [4]:
import pandas as pd

# Load the raw pickled data
pkl_path = "data/processed/geography.pkl"
pdf = pd.read_pickle(pkl_path)


# Print information to inspect schemas visually
print("--- Pandas Dataframe Head ---")
print(pdf.head(3))
print("\n--- Available Columns ---")
print(pdf.columns.tolist())

--- Pandas Dataframe Head ---
      Country    district  Ind_adm2_ID  point_order           state  \
40956   India  Srikakulam           20         5249  Andhra Pradesh   
40957   India  Srikakulam           20         5250  Andhra Pradesh   
40958   India  Srikakulam           20         5251  Andhra Pradesh   

       sub_polygon_id  Latitude  Longitude  Number of Records  
40956               1   18.2193    83.9543                  1  
40957               1   18.2190    83.9543                  1  
40958               1   18.2190    83.9535                  1  

--- Available Columns ---
['Country', 'district', 'Ind_adm2_ID', 'point_order', 'state', 'sub_polygon_id', 'Latitude', 'Longitude', 'Number of Records']


In [5]:
# Convert mixed columns to strings to ensure clean Spark ingestion
pdf_clean = pdf.astype(str)
geo_raw_spark = spark.createDataFrame(pdf_clean)

# Deduplicate to establish the master dimension grain
geo_dim_df = geo_raw_spark.select(
    col("Country").alias("country"),
    col("state").alias("state"),
    col("district").alias("district")
).distinct()

# Generate the unique SHA-256 Surrogate Key for joins
geo_dim_df = geo_dim_df.withColumn(
    "geography_key", 
    sha2(concat_ws("||", col("country"), col("state"), col("district")), 256)
)

# Display the first few rows to verify the transformation layout
print("--- Master Geography Dimension Layout ---")
geo_dim_df.show(5, truncate=False)

/home/abhishek/dsba2026/dsba/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/abhishek/dsba2026/dsba/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


--- Master Geography Dimension Layout ---


+-------+--------------+------------+----------------------------------------------------------------+
|country|state         |district    |geography_key                                                   |
+-------+--------------+------------+----------------------------------------------------------------+
|India  |Andhra Pradesh|Nellore     |f5b5fd5bea97f703b80282beb87bb6b3d616b6e344426c2170921f10ee69be88|
|India  |Andhra Pradesh|Vizianagaram|a23777f21a6ccb5eba191f80a06ae84953ab5530875f8cd6f20fcebedb5ef5b4|
|India  |Andhra Pradesh|Srikakulam  |3d9160d09d8ab74a900924666b849511b83397676e64e6508e297a209106337c|
|India  |Andhra Pradesh|Prakasam    |c35bc0e7f290a249be0b94a105f65e7c57e95a19965e24337d975109380412a6|
|India  |Andhra Pradesh|Warangal    |fbe464d663220be932b5ea8f4dadc690b444e564a03104c271ef6641e8557d66|
+-------+--------------+------------+----------------------------------------------------------------+
only showing top 5 rows


In [15]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
# Explicitly force the import from PySpark to overwrite any wrong local bindings
from pyspark.sql.functions import col, sha2

# 1. Define the explicit schema for our seasons
season_schema = StructType([
    StructField("season_name", StringType(), False),
    StructField("start_month", IntegerType(), False),
    StructField("end_month", IntegerType(), False)
])

# 2. Hardcode the standard structural mappings
season_data = [
    ("Kharif", 6, 10),
    ("Rabi", 11, 4),
    ("Summer", 5, 6),
    ("Autumn", 9, 11),
    ("Winter", 12, 2),
    ("Whole Year", 1, 12),
    ("Total", 1, 12)
]

# 3. Create the dataframe and generate a surrogate hash key
season_raw_df = spark.createDataFrame(season_data, schema=season_schema)

dim_season_df = season_raw_df.withColumn(
    "season_key",
    sha2(col("season_name"), 256)
)

dim_season_df.show(10, truncate=False)


+-----------+-----------+---------+----------------------------------------------------------------+
|season_name|start_month|end_month|season_key                                                      |
+-----------+-----------+---------+----------------------------------------------------------------+
|Kharif     |6          |10       |7e7fabd8340e7201c8ce2e678b32f80b3e8febdd6d10900ce788be0f9fb1c33e|
|Rabi       |11         |4        |8b3a775b0b4376bb09ef2cbadd285789f26136cbf3d2543213ed3b8c2ab6dbbc|
|Summer     |5          |6        |c0de243c407c0e98ed0364e1d0c636d542318867862aac7e8ae2446d4a328dd6|
|Autumn     |9          |11       |e0906c2d6dc6d5263762d7ee5895e0544c326c26fb7f4f3163cde2d20b8f068f|
|Winter     |12         |2        |1b17869e03ecc1b0f86b003c199606eb68eaceb18927e3843a935cf7f6157152|
|Whole Year |1          |12       |ab06ef1e97363d61fb3eeb8e08085d89dc5057d2b49b516b332dbe374874edf6|
|Total      |1          |12       |c9b3c38247f744e17dd26fda097d6a9ba9332586b6bdaa038bf8f313

In [16]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# 1. Define the explicit schema for our seasons
season_schema = StructType([
    StructField("season_name", StringType(), False),
    StructField("start_month", IntegerType(), False),
    StructField("end_month", IntegerType(), False)
])

# 2. Hardcode the standard structural mappings
season_data = [
    ("Kharif", 6, 10),
    ("Rabi", 11, 4),
    ("Summer", 5, 6),
    ("Autumn", 9, 11),
    ("Winter", 12, 2),
    ("Whole Year", 1, 12),
    ("Total", 1, 12)
]

# 3. Create the dataframe and generate a surrogate hash key
season_raw_df = spark.createDataFrame(season_data, schema=season_schema)

dim_season_df = season_raw_df.withColumn(
    "season_key",
    sha2(col("season_name"), 256)
)

dim_season_df.show(10, truncate=False)

+-----------+-----------+---------+----------------------------------------------------------------+
|season_name|start_month|end_month|season_key                                                      |
+-----------+-----------+---------+----------------------------------------------------------------+
|Kharif     |6          |10       |7e7fabd8340e7201c8ce2e678b32f80b3e8febdd6d10900ce788be0f9fb1c33e|
|Rabi       |11         |4        |8b3a775b0b4376bb09ef2cbadd285789f26136cbf3d2543213ed3b8c2ab6dbbc|
|Summer     |5          |6        |c0de243c407c0e98ed0364e1d0c636d542318867862aac7e8ae2446d4a328dd6|
|Autumn     |9          |11       |e0906c2d6dc6d5263762d7ee5895e0544c326c26fb7f4f3163cde2d20b8f068f|
|Winter     |12         |2        |1b17869e03ecc1b0f86b003c199606eb68eaceb18927e3843a935cf7f6157152|
|Whole Year |1          |12       |ab06ef1e97363d61fb3eeb8e08085d89dc5057d2b49b516b332dbe374874edf6|
|Total      |1          |12       |c9b3c38247f744e17dd26fda097d6a9ba9332586b6bdaa038bf8f313

In [ ]:
# Point directly to the folder name, not a specific file
dim_date_df = spark.read.parquet("data/warehouse/dim_date")

# Show the first 5 records
dim_date_df.show(5, truncate=False)

dim_geography_df = spark.read.parquet("data/warehouse/dim_geography")

# Show the first 5 records
dim_geography_df.show(5, truncate=False)

dim_season_df = spark.read.parquet("data/warehouse/dim_season")

# Show the first 5 records
dim_season_df.show(5, truncate=False)


+----------+------------+----+-----+----------+---+-----------+---------+-------+----------+
|date_key  |display_date|year|month|month_name|day|day_of_week|day_name |quarter|is_weekend|
+----------+------------+----+-----+----------+---+-----------+---------+-------+----------+
|2006-09-17|17-09-2006  |2006|9    |September |17 |1          |Sunday   |3      |1         |
|2006-09-18|18-09-2006  |2006|9    |September |18 |2          |Monday   |3      |0         |
|2006-09-19|19-09-2006  |2006|9    |September |19 |3          |Tuesday  |3      |0         |
|2006-09-20|20-09-2006  |2006|9    |September |20 |4          |Wednesday|3      |0         |
|2006-09-21|21-09-2006  |2006|9    |September |21 |5          |Thursday |3      |0         |
+----------+------------+----+-----+----------+---+-----------+---------+-------+----------+
only showing top 5 rows
+-------+--------------+------------+----------------------------------------------------------------+
|country|state         |district    

26/11/03 10:17:10 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-----------+-----------------+-----------------+----------------------------------------------------------------+
|summary|season_name|start_month      |end_month        |season_key                                                      |
+-------+-----------+-----------------+-----------------+----------------------------------------------------------------+
|count  |7          |7                |7                |7                                                               |
|mean   |NULL       |6.428571428571429|8.142857142857142|NULL                                                            |
|stddev |NULL       |4.466808810815721|4.099941927579439|NULL                                                            |
|min    |Autumn     |1                |2                |1b17869e03ecc1b0f86b003c199606eb68eaceb18927e3843a935cf7f6157152|
|max    |Winter     |12               |12               |e0906c2d6dc6d5263762d7ee5895e0544c326c26fb7f4f3163cde2d20b8f068f|
+-------+-------